In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gc
sns.set_theme()
gc.enable()

# Aggregating Bureau Balance

In [2]:
bureau_balance=pd.read_csv('bureau_balance.csv')
bureau_balance.head()

,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C


In [3]:
bureau_balance.shape

(27299925, 3)

In [5]:
bureau_balance['STATUS'].unique()

array(['C', '0', 'X', '1', '2', '3', '5', '4'], dtype=object)

In [33]:
bureau_balance_agg=bureau_balance[['SK_ID_BUREAU', 'STATUS']].value_counts().unstack().replace(np.nan, 0).reset_index()
bureau_balance_agg.head()

STATUS,SK_ID_BUREAU,0,1,2,3,4,5,C,X
0,5001709,0.0,0.0,0.0,0.0,0.0,0.0,86.0,11.0
1,5001710,5.0,0.0,0.0,0.0,0.0,0.0,48.0,30.0
2,5001711,3.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,5001712,10.0,0.0,0.0,0.0,0.0,0.0,9.0,0.0
4,5001713,0.0,0.0,0.0,0.0,0.0,0.0,0.0,22.0


In [34]:
bureau_balance_agg['TOTAL']=bureau_balance_agg.drop('SK_ID_BUREAU', axis=1).sum(axis=1)
bureau_balance_agg.head()

STATUS,SK_ID_BUREAU,0,1,2,3,4,5,C,X,TOTAL
0,5001709,0.0,0.0,0.0,0.0,0.0,0.0,86.0,11.0,97.0
1,5001710,5.0,0.0,0.0,0.0,0.0,0.0,48.0,30.0,83.0
2,5001711,3.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4.0
3,5001712,10.0,0.0,0.0,0.0,0.0,0.0,9.0,0.0,19.0
4,5001713,0.0,0.0,0.0,0.0,0.0,0.0,0.0,22.0,22.0


In [35]:
bureau_balance_agg.rename(columns={'0': 'STATUS_NO_DPD', '1': 'STATUS_<31DPD', '2': 'STATUS_<61DPD', '3': 'STATUS_<91DPD', 
                                   '4': 'STATUS_<121DPD', '5': 'STATUS_>120DPD',
                                   'C': 'STATUS_CLOSED', 'X': 'STATUS_NA', 'TOTAL': 'STATUS_TOTAL'}, inplace=True)
bureau_balance_agg.head()

STATUS,SK_ID_BUREAU,STATUS_NO_DPD,STATUS_<31DPD,STATUS_<61DPD,STATUS_<91DPD,STATUS_<121DPD,STATUS_>120DPD,STATUS_CLOSED,STATUS_NA,STATUS_TOTAL
0,5001709,0.0,0.0,0.0,0.0,0.0,0.0,86.0,11.0,97.0
1,5001710,5.0,0.0,0.0,0.0,0.0,0.0,48.0,30.0,83.0
2,5001711,3.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4.0
3,5001712,10.0,0.0,0.0,0.0,0.0,0.0,9.0,0.0,19.0
4,5001713,0.0,0.0,0.0,0.0,0.0,0.0,0.0,22.0,22.0


# Dealing with main Bureau data

In [36]:
del bureau_balance
bureau=pd.read_csv('bureau.csv')
bureau.head()

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


#### Merging with aggregated bureau balance data

In [40]:
bureau['SK_ID_BUREAU'].unique().shape, bureau_balance_agg['SK_ID_BUREAU'].unique().shape

((1716428,), (817395,))

In [43]:
combined_data=pd.merge(left=bureau, right=bureau_balance_agg, how='left', on='SK_ID_BUREAU')
combined_data.head()

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,...,AMT_ANNUITY,STATUS_NO_DPD,STATUS_<31DPD,STATUS_<61DPD,STATUS_<91DPD,STATUS_<121DPD,STATUS_>120DPD,STATUS_CLOSED,STATUS_NA,STATUS_TOTAL
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [49]:
numerical_features=combined_data.select_dtypes(exclude='object').columns
categorical_features=combined_data.columns.difference(numerical_features)

## Numerical Features

In [50]:
numerical_frame=combined_data[numerical_features].copy()
numerical_frame.head()

,SK_ID_CURR,SK_ID_BUREAU,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,...,AMT_ANNUITY,STATUS_NO_DPD,STATUS_<31DPD,STATUS_<61DPD,STATUS_<91DPD,STATUS_<121DPD,STATUS_>120DPD,STATUS_CLOSED,STATUS_NA,STATUS_TOTAL
0,215354,5714462,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,215354,5714463,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,215354,5714464,-203,0,528.0,NaN,NaN,0,464323.5,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,215354,5714465,-203,0,NaN,NaN,NaN,0,90000.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,215354,5714466,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [51]:
numerical_frame.isnull().sum()

SK_ID_CURR                      0
SK_ID_BUREAU                    0
DAYS_CREDIT                     0
CREDIT_DAY_OVERDUE              0
DAYS_CREDIT_ENDDATE        105553
DAYS_ENDDATE_FACT          633653
AMT_CREDIT_MAX_OVERDUE    1124488
CNT_CREDIT_PROLONG              0
AMT_CREDIT_SUM                 13
AMT_CREDIT_SUM_DEBT        257669
AMT_CREDIT_SUM_LIMIT       591780
AMT_CREDIT_SUM_OVERDUE          0
DAYS_CREDIT_UPDATE              0
AMT_ANNUITY               1226791
STATUS_NO_DPD              942074
STATUS_<31DPD              942074
STATUS_<61DPD              942074
STATUS_<91DPD              942074
STATUS_<121DPD             942074
STATUS_>120DPD             942074
STATUS_CLOSED              942074
STATUS_NA                  942074
STATUS_TOTAL               942074
dtype: int64

In [52]:
numerical_frame['SK_ID_CURR'].unique().shape

(305811,)